In [1]:
from sentence_transformers import SentenceTransformer

from datasets import load_from_disk

import faiss

import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
wiki = load_from_disk(
    "../datasets/Wikipedia"
)

In [3]:
index = faiss.read_index(
    "../faiss_index/wiki.index"
)

In [4]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\SujanRam\OneDrive\Documents\Hallucination project\.venv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
def retrieve_evidence(query, top_k=5):

    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    passages = []

    scores = []

    retrieved_embeddings = []

    for idx in indices[0]:

        article = wiki[int(idx)]

        passage = article["text"][:500]

        passages.append(passage)

        retrieved_embeddings.append(
            model.encode(passage)
        )

    retrieved_embeddings = np.array(
        retrieved_embeddings
    )

    sims = cosine_similarity(
        query_embedding,
        retrieved_embeddings
    )[0]

    scores = sims.tolist()

    return passages, scores

In [7]:
claim = "Paris is the capital of Germany"

In [8]:
passages, scores = retrieve_evidence(
    claim,
    top_k=5
)

In [9]:
for i in range(len(passages)):

    print("\n====================")

    print("Evidence", i + 1)

    print("====================")

    print("Similarity Score:", scores[i])

    print(passages[i][:700])


Evidence 1
Similarity Score: 0.5602269172668457
Villeparisis () is a commune in the Seine-et-Marne department in the Île-de-France region in north-central France. It is located in the north-eastern suburbs of Paris  from the centre.

Inhabitants of Villeparisis are called Villeparisiens.

Population

Transport
Villeparisis is served by Villeparisis–Mitry-le-Neuf station on Paris RER line B.

Twin towns – sister cities

Villeparisis is twinned with:
 Maldon, England, United Kingdom
 Pietrasanta, Italy
 Wathlingen, Germany

Notable people
Henri

Evidence 2
Similarity Score: 0.5517727136611938
Bonn () is a federal city in the German state of North Rhine-Westphalia, located on the banks of the Rhine. It has a population of over 300,000. About  south-southeast of Cologne, Bonn is in the southernmost part of the Rhine-Ruhr region, Germany's largest metropolitan area, with over 11 million inhabitants. It is a university city, was the birthplace of Ludwig van Beethoven and was the capital of 